# Week 9 — Feedback Intelligence & Model Refinement
**Tools:** Streamlit Cloud, Pandas, Plotly, Gemini API
**Data:** All module outputs → feedback CSV → analytics → model refinement loop

In [ ]:
!pip install pandas numpy plotly google-generativeai -q
import pandas as pd,numpy as np,json,re,plotly.express as px
from datetime import datetime
from pathlib import Path
print('Libraries loaded ✓')

In [ ]:
# Simulate feedback data (in production: loaded from Streamlit-generated CSV)
np.random.seed(42)
n=120
fb=pd.DataFrame({'timestamp':pd.date_range('2025-01-01',periods=n,freq='3H').astype(str),'company':np.random.choice(['NovaTech','AlphaStart','GreenLeaf','PixelCo','TechNova'],n),'industry':np.random.choice(['Technology','Healthcare','Fashion','Finance','Education'],n),'brand_tone':np.random.choice(['Tech-Forward & Innovative','Minimalist','Bold & Vibrant'],n),'logo_rating':np.random.randint(2,6,n),'tagline_rating':np.random.randint(2,6,n),'campaign_rating':np.random.randint(2,6,n),'multilingual_rating':np.random.randint(2,6,n),'comments':np.random.choice(['Great taglines!','Logo looks premium','Need more colour variety','Campaign post was helpful','Font suggestion was perfect','Translation sounded very natural','Could use more industry-specific taglines'],n)})
fb['avg_rating']=fb[['logo_rating','tagline_rating','campaign_rating','multilingual_rating']].mean(axis=1).round(2)
fb.to_csv('feedback_data.csv',index=False)
print(f'Feedback dataset: {fb.shape} | Avg rating: {fb["avg_rating"].mean():.2f}/5')

In [ ]:
# Feedback analytics
rating_cols=['logo_rating','tagline_rating','campaign_rating','multilingual_rating']
avgs=fb[rating_cols].mean().reset_index(); avgs.columns=['Module','Avg Rating']
avgs['Module']=avgs['Module'].str.replace('_rating','').str.replace('_',' ').str.title()
fig=px.bar(avgs,x='Module',y='Avg Rating',color='Avg Rating',color_continuous_scale='Purples',title=f'Average Ratings by Module ({len(fb)} submissions)',text=avgs['Avg Rating'].round(2).astype(str)+'/5')
fig.update_layout(yaxis_range=[0,5],plot_bgcolor='white'); fig.update_traces(textposition='outside'); fig.show()

In [ ]:
# Rating by brand tone
tone_ratings=fb.groupby('brand_tone')['avg_rating'].mean().reset_index().sort_values('avg_rating',ascending=False)
fig2=px.bar(tone_ratings,x='brand_tone',y='avg_rating',title='Avg Rating by Brand Tone',color='avg_rating',color_continuous_scale='Blues')
fig2.update_layout(plot_bgcolor='white',yaxis_range=[0,5]); fig2.show()
# Rating over time trend
fb['date']=pd.to_datetime(fb['timestamp']).dt.date
daily=fb.groupby('date')['avg_rating'].mean().reset_index()
fig3=px.line(daily,x='date',y='avg_rating',title='Daily Avg Rating Trend',markers=True)
fig3.update_layout(plot_bgcolor='white',yaxis_range=[0,5]); fig3.show()

In [ ]:
# Gemini sentiment analysis on comments
import google.generativeai as genai
from google.colab import userdata
genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
gm=genai.GenerativeModel('gemini-1.5-flash')
comments=fb['comments'].unique().tolist()
p=f"""Analyse the sentiment of these user feedback comments for an AI branding tool.\n\nComments: {comments}\n\nReturn ONLY valid JSON:\n{{\"overall_sentiment\":\"positive/neutral/negative\",\"key_themes\":[\"t1\",\"t2\",\"t3\"],\"top_praised\":\"...\",\"improvement_areas\":[\"a1\",\"a2\"],\"recommendation\":\"...\"}}"""
try:
    raw=gm.generate_content(p).text.strip(); m=re.search(r'\{.*?\}',raw,re.DOTALL)
    if m: print('Feedback Sentiment Analysis:'); print(json.dumps(json.loads(m.group()),indent=2))
except Exception as e: print('Add Gemini key to run:',e)

In [ ]:
# Module-level feedback summary
print('\n=== Feedback Intelligence Summary ===')
print(f'Total submissions  : {len(fb)}')
print(f'Overall avg rating : {fb["avg_rating"].mean():.2f} / 5')
print(f'Best-rated module  : {avgs.loc[avgs["Avg Rating"].idxmax(),"Module"]}')
print(f'Needs improvement  : {avgs.loc[avgs["Avg Rating"].idxmin(),"Module"]}')
print(f'Top brand tone     : {tone_ratings.iloc[0]["brand_tone"]}')
# Save report
report={'total_submissions':len(fb),'overall_avg':round(float(fb['avg_rating'].mean()),2),'module_avgs':avgs.set_index('Module')['Avg Rating'].to_dict(),'by_tone':tone_ratings.set_index('brand_tone')['avg_rating'].to_dict()}
with open('week9_feedback_report.json','w') as f: json.dump(report,f,indent=2)
print('\nSaved week9_feedback_report.json ✓')

## ✅ Week 9 Deliverables
- [ ] Feedback data (120 simulated submissions) created and saved as feedback_data.csv
- [ ] Rating analytics: bar chart, tone breakdown, daily trend
- [ ] Gemini sentiment analysis on feedback comments
- [ ] Module-level performance summary
- [ ] Saved: week9_feedback_report.json

## Week 10 Deployment Checklist

### Push to GitHub
```bash
git init
git add .
git commit -m 'BrandSphere AI — NovaTech CRS Capstone 2025-26'
git remote add origin https://github.com/<username>/IADAI205-<IDs>.git
git push -u origin main
```

### Streamlit Cloud Deployment
1. https://streamlit.io/cloud → New App
2. Select repo → Main file: `streamlit_app/app.py`
3. Advanced settings → Add secret: `GEMINI_API_KEY = "your_key"`
4. Deploy → live in ~2 minutes

### GitHub Repo Name Format
`IADAI205-[ID1]-[Name1]-[ID2]-[Name2]-[ID3]-[Name3]-[ID4]-[Name4]`